# Running TRL methods

The toolkit wraps several [TRL](https://github.com/huggingface/trl) trainers as structural controls. In this guide we fine-tune a small model on preference data through a `SteeringPipeline`. We cover supervised fine-tuning (SFT) and direct preference optimization (DPO) with LoRA adapters, anchored preference optimization (APO) as a DPO-family variant, and a full-parameter SFT run. We also show how to resume an interrupted run from a checkpoint and how to serve the trained artifact (a merged checkpoint or a LoRA adapter) on the vLLM backend.

## Setup

If running this from a Google Colab notebook, please uncomment the following cell to install the toolkit. The following block is not necessary if running this notebook from a virtual environment where the package has already been installed.

In [ ]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [ ]:
# !pip install python-dotenv
# !pip install ipywidgets
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

Next, we import the `SteeringPipeline` class (used throughout) and specify the base model, in this case a small Qwen model.

In [3]:
import torch
from datasets import load_dataset
from peft import PeftType
from transformers import AutoTokenizer

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Data Preparation

The controls throughout this notebook are trained using a common dataset, `ultrafeedback_binarized`, since it contains preference data for each prompt (which is necessary for DPO-based controls). We load each of the splits below.

In [4]:
raw_train = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
raw_test  = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs")
len(raw_train), raw_train[0].keys()

(61135,
 dict_keys(['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected']))

Different trainers expect different data formats (i.e., tensor layouts) and thus we define two helper functions, one for SFT and one for DPO, to process the data in a way that is amenable to each.

In [5]:
def sft_preprocess(example, tokenizer, max_length=1024):
    answer = example["chosen"][-1]["content"]
    text = f"Question: {example['prompt']}\n\nAnswer: {answer}"
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_length)
    labels = [
        token_id if mask == 1 else -100  # label pads as -100 so they don't contribute to loss
        for token_id, mask in zip(encoding["input_ids"], encoding["attention_mask"])
    ]
    encoding["labels"] = labels
    return encoding

def dpo_filter(example, max_prompt_chars=4000):
    return {
        "prompt": example["prompt"][:max_prompt_chars],
        "chosen": example["chosen"][-1]["content"],
        "rejected": example["rejected"][-1]["content"],
    }


subset_size = 500

sft_train = raw_train.select(range(subset_size)).map(
    lambda example: sft_preprocess(example, tokenizer, max_length=1024),
    remove_columns=raw_train.column_names
)

dpo_train = raw_train.select(range(subset_size)).map(dpo_filter, remove_columns=[])
dpo_train[0].keys()

Map:   0%|                                                                                                                                                                    | 0/500 [00:00<?, ? examples/s]

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 14842.37 examples/s]


dict_keys(['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'])

## SFT control

We now show how to fine-tune with SFT using LoRA. We also merge the trained adapter back into the model (using the argument `merge_lora_after_train`). Note the argument `use_peft=True` to indicate that we are not running a full fine-tune (the example near the end of this notebook will illustrate a full fine-tuning run). 

In [6]:
from aisteer360.algorithms.structural_control.wrappers.trl.sfttrainer.control import SFT


sft = SFT(
    # data
    train_dataset=sft_train,
    eval_dataset=None, 
    # data_collator=None  # optional; if omitted and you provided labels, you're fine

    # TRL / Trainer config (forwarded into SFTConfig)
    output_dir="./tmp/sft_lora",
    max_seq_length=1024,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=50,
    report_to="none",
    seed=42,

    # PEFT (LoRA)
    use_peft=True,
    peft_type=PeftType.LORA,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    adapter_name="sft",

    # optionally merge LoRA into base weights after training
    merge_lora_after_train=True,
    merged_output_dir="./tmp/sft_lora_merged",
)


We create a steering pipeline using the above control, without a `model_name_or_path` since the structural control (`sft`) returns a model. The pipeline is then steered which invokes the training procedure.

In [7]:
sft_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    device_map=None,
    hf_model_kwargs={"trust_remote_code": True},
    controls=[sft],
)

sft_pipeline.steer()


Truncating train dataset: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 23946.65 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,1.675000
100,1.614800


The above SFT-trained pipeline is now ready for inference.

In [8]:
prompt = "Question: What makes the sky look blue?\n\nAnswer:"
print(sft_pipeline.generate(prompt, max_new_tokens=64))

 The sky appears to be blue because of the scattering of sunlight by tiny water droplets in the atmosphere. This process is called Rayleigh scattering, and it occurs at different wavelengths of light depending on the size of the droplet. When sunlight enters a cloud or fog, some of the shorter-wavelength (blue) light


## DPO control

DPO is instantiated in a similar fashion with the primary differences being that the training data is now triples (`prompt`, `chosen`, `rejected`), the trainer must keep a reference policy alongside the trainable policy, and the loss is a pair-wise KL-reg. contrastive objective rather than the token-level cross entropy loss in SFT. 

Note: By default, the trainer clones the base weights and freezes them. When LoRA is enabled, the wrapper automatically passes `ref_model=None`, letting TRL re-create a frozen reference that shares the same LoRA adapters. If you are full fine-tuning you can still supply your own `ref_model` via `pipeline.steer(ref_model=my_frozen_model)`.

In [9]:
from aisteer360.algorithms.structural_control.wrappers.trl.dpotrainer.control import DPO


dpo = DPO(
    train_dataset=dpo_train,

    # DPO / TRL config (forwarded into DPOConfig)
    output_dir="./tmp/dpo_lora",
    per_device_train_batch_size=2,  # often smaller than SFT
    num_train_epochs=1,
    learning_rate=1e-5,
    beta=0.1,
    loss_type="sigmoid",  # baseline DPO loss
    max_prompt_length=512,
    max_length=1024,
    precompute_ref_log_probs=False,  # off: avoids the noisy per-batch reference log-prob pass; enable for multi-epoch runs where the precompute is reused
    disable_dropout=True,
    logging_steps=50,
    report_to="none",
    seed=123,

    # LoRA
    use_peft=True,
    peft_type=PeftType.LORA,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    adapter_name="dpo",

    merge_lora_after_train=False,
)

As before, we create the pipeline using the control, steer the pipeline, and run inference on the steered pipeline.

In [10]:
dpo_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    hf_model_kwargs={"trust_remote_code": True},
    controls=[dpo]
)
dpo_pipeline.steer()


Tokenizing train dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 801.53 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
50,0.686600
100,0.696300
150,0.693700
200,0.695000
250,0.701900


In [11]:
prompt = "Question: Is it ever helpful to be blunt with feedback?\n\nAnswer:"
print(dpo_pipeline.generate(prompt, max_new_tokens=150))

 Yes, it is always helpful to be blunt with feedback. Blunt feedback can help you identify areas of improvement and provide a clear path for change. It also helps to build trust between the person being evaluated and the person giving the feedback.

For example, if someone gives you feedback that says "You need to improve your writing skills," you could respond by saying "I agree, but I think we should focus on improving our research methods instead." This response provides constructive criticism without sounding accusatory or dismissive.

Blunt feedback can also help to motivate people to take action towards their goals. If someone gives you feedback that says "You need to work harder on this project," you could say "Thank you for your input, but I think we can


## APO control

APO lives in the same trainer family as DPO and uses the same `DPOTrainer` class (it is activated by simply choosing a different `loss_type`). In contrast to DPO that pushes the policy away from the reference (by a relative KL-scaled margin), APO pushes the policy toward a fixed "anchor" score. Generally, APO keeps the policy closer to the reference for the same beta, reducing the risk of over-optimization.

In [ ]:
from aisteer360.algorithms.structural_control.wrappers.trl.apotrainer.control import APO


apo = APO(
    # data
    train_dataset=dpo_train,

    # APO / TRL config 
    output_dir="./tmp/apo_lora",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=1e-5,
    beta=0.1,
    loss_type="apo_zero",     # APO-specific loss
    max_prompt_length=512,
    max_length=1024,
    precompute_ref_log_probs=False,  # inherited default is True (APOArgs subclasses DPOArgs); off for the same reason as the DPO cell
    logging_steps=50,
    report_to="none",
    seed=99,

    # LoRA
    use_peft=True,
    peft_type=PeftType.LORA,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    adapter_name="apo",
    
    merge_lora_after_train=False,
)


Steering and inference proceeds as before.

In [13]:
apo_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    hf_model_kwargs={"trust_remote_code": True},
    controls=[apo]
)
apo_pipeline.steer()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
50,1.003700
100,1.001700
150,1.001200
200,0.999300
250,0.998500


In [14]:
prompt = "Question: Explain why kindness can be strategic.\n\nAnswer:"
print(apo_pipeline.generate(prompt, max_new_tokens=64))

 Kindness is a powerful tool that can be used strategically in various situations. It allows us to connect with others, build trust and relationships, and promote positive change. By being kind, we can create a positive impact on the world and help others in need. Additionally, kindness can be used as a way to set an


## Full-parameter SFT

Lastly, to run a full-weight fine-tune set `use_peft=False`, drop the LoRA arguments, and usually shrink the batch size (because every parameter now receives gradients). 

Note: Full fine-tuning can be 10-20 times more memory-intensive than LoRA.

In [15]:
full_sft = SFT(
    train_dataset=sft_train,
    use_peft=False,  # full FT
    output_dir="./tmp/sft_full",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    learning_rate=5e-6,
    report_to="none",
    seed=7,
)
full_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    hf_model_kwargs={"trust_remote_code": True},
    controls=[full_sft]
)
full_pipeline.steer()


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.475400
20,1.754500
30,1.795500
40,1.913600
50,1.726500
60,1.653900
70,1.518100
80,1.626200
90,1.310000
100,1.771500


The wrapper also provides functionality for resuming training if interrupted (via TRL's `resume_from_checkpoint`) by providing either the directory path of the checkpoint name in `output_dir`.

In [16]:
resume_sft = SFT(
    train_dataset=sft_train,
    output_dir="./tmp/sft_lora",
    resume_from_checkpoint="./tmp/sft_lora/checkpoint-1000",
    use_peft=True,
    adapter_name="sft",
    report_to="none",
)
resume_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    hf_model_kwargs={"trust_remote_code": True},
    controls=[resume_sft]
)
resume_pipeline.steer()


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.654300
20,1.742200
30,1.639700
40,1.604800
50,1.573300
60,1.712700
70,1.798500
80,1.620000
90,1.616700
100,1.788600


## Serving the trained artifact on vLLM

Structural controls train on live weights, so on an engine backend the steer phase runs on a temporary in-process model (the stage) that is freed before the engine boots. The exported artifact carries the training across to the engine. A full fine-tune or a merged LoRA run exports a checkpoint (`CheckpointArtifact`), which overrides the model the engine serves. A LoRA run without merging exports the adapter (`LoRAArtifact`) instead, which the engine attaches as a LoRA request (`enable_lora` is set for you). No plugin is involved since the artifact is plain weights, so any vLLM install serves it. Note that running this section requires the toolkit's `vllm` extra, and the `vllm-serve` backend works the same way against a running server.

We rerun the earlier LoRA SFT configuration with fresh output directories inside a single pipeline whose backend is the offline engine. The `steer()` call trains on the staged model exactly as before, and generation then runs on vLLM serving the merged checkpoint. With `merge_lora_after_train=False` the engine would serve the base model with the adapter attached instead.

In [17]:
from aisteer360.algorithms.core.execution import BackendSpec

sft_vllm = SFT(
    # data
    train_dataset=sft_train,

    # TRL / Trainer config (forwarded into SFTConfig)
    output_dir="./tmp/sft_lora_vllm",
    max_seq_length=1024,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=50,
    report_to="none",
    seed=42,

    # PEFT (LoRA)
    use_peft=True,
    peft_type=PeftType.LORA,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    adapter_name="sft",

    # merge so the exported artifact is a checkpoint the engine serves directly
    merge_lora_after_train=True,
    merged_output_dir="./tmp/sft_lora_vllm_merged",
)

torch.cuda.empty_cache()

engine_spec = BackendSpec(
    kind="vllm",
    model=MODEL_NAME,
    options={
        "trust_remote_code": True,
        "engine_kwargs": {"gpu_memory_utilization": 0.35, "max_model_len": 2048},
    },
)

with SteeringPipeline(
    controls=[sft_vllm],
    backend=engine_spec,
    hf_model_kwargs={"trust_remote_code": True},
) as vllm_pipeline:
    vllm_pipeline.steer()
    prompt = "Question: What makes the sky look blue?\n\nAnswer:"
    print(vllm_pipeline.generate(prompt, max_new_tokens=64, do_sample=False))

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,1.675200
100,1.614300


The tokenizer you are loading from './tmp/sft_lora_vllm_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:07<00:00,  7.65s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:07<00:00,  7.65s/it]
(EngineCore pid=1156070) 
(EngineCore pid=1156070) 2026-08-20 17:23:43,626 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=1156070) 2026-08-20 17:23:43,638 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=1156070) The tokenizer you are loading from './tmp/sft_lora_vllm_merged' with an incorrect regex pat

 The sky looks blue because it is made up of tiny particles of gas and dust that scatter light. These particles are called "air molecules" and they are responsible for the color of the sky. The blue color of the sky is due to the fact that the air molecules scatter blue light more than other colors of light.


[rank0]:[W820 17:23:48.027633003 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


The `pipeline.check()` method reports this plan before any work happens. The SFT step is `module` access and runs on the stage, and the generate phase is supported because the configuration exports a servable checkpoint. A LoRA configuration without an output directory would instead be in-process only, and the verdict says so. The staged weights are freed before the engine boots, so the trained copy and the served copy never coexist, and exiting the `with` block shuts the engine down. Note that the offline engine's release is process-global with respect to vLLM's distributed state, so run this section with no other live vLLM engine in the process.